# 🧠 NioMini (~7.1M) Defect Forecaster - Google Colab Training Pipeline

**NioMini** is a pre-emptive defect forecasting engine designed to analyze code diffs and predict:
1. **Build Break Verdict**: Binary classification whether the commit/diff breaks the build (`will_break`)
2. **Defect Category**: Multi-class classification (`CLEAN`, `SYNTAX_ERROR`, `TYPE_MISMATCH`, `BORROW_CHECK_FAIL`, `TEST_REGRESSION`)
3. **Calibrated Risk Score**: Continuous probability score $[0.0, 1.0]$

> **Exact Parameter Budget**: `7,100,000` parameters (enforced by architecture invariant).
> **License**: Business Source License 1.1 (BSL 1.1) converting to Apache 2.0.

## 1. Environment & GPU Verification
Run on Google Colab with a free **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 2. Clone Workspace or Install Dependencies

In [ ]:
!pip install -q onnx onnxruntime onnxscript numpy

import os
if not os.path.exists("nio-xs-models"):
    !git clone https://github.com/nio-labs/nio-xs-models.git
    %cd nio-xs-models/models/nio-mini/scripts
else:
    %cd nio-xs-models/models/nio-mini/scripts

!ls -la

## 3. Verify Model Parameter Budget (`7,100,000` parameters)

In [ ]:
from model import NioMiniDefectForecaster, NioMiniConfig

config = NioMiniConfig()
model = NioMiniDefectForecaster(config)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"NioMini Model Parameters: {param_count:,}")
assert param_count == 7_100_000, f"Expected 7,100,000 parameters, got {param_count}"
print("✅ Parameter invariant passed!")

## 4. Run Model Training & Validation
Generates realistic AST-annotated code samples across all 5 defect categories and trains the model.

In [ ]:
!python train.py --samples 15000 --epochs 5 --batch-size 64 --onnx-out /content/nio-mini.onnx

## 5. Test Inference with ONNX Runtime

In [ ]:
from infer import NioMiniPredictor

predictor = NioMiniPredictor("/content/nio-mini.onnx")

test_samples = [
    ("Clean Code", "pub fn add(a: i32, b: i32) -> i32 { a + b }"),
    ("Syntax Error", "pub fn broken() { let x = 1;"),
    ("Borrow Conflict", "pub fn borrow_fail(v: &mut Vec<i32>) { let a = &mut v; let b = &mut v; }"),
    ("Type Mismatch", "error[E0308]: mismatched types: expected `u64`, found `i32`"),
    ("Test Regression", "#[test]\nfn test_regression() { assert!(false); }"),
]

for label, code in test_samples:
    verdict = predictor.predict(code)
    status = "❌ BREAK" if verdict["will_break"] else "✅ CLEAN"
    print(f"[{status}] {label} -> Predicted: {verdict['category']} (Break: {verdict['break_probability']*100:.1f}%, Risk: {verdict['risk_score']:.2f})")

## 6. Download Trained ONNX Model

In [ ]:
from google.colab import files
import os

if os.path.exists("/content/nio-mini.onnx"):
    files.download("/content/nio-mini.onnx")
    print("Downloaded /content/nio-mini.onnx")
else:
    print("Model file not found at /content/nio-mini.onnx")